In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 작업디렉토리 변경
%cd "/content/drive/MyDrive/AI/인사교_LangChain_20260624"

/content/drive/MyDrive/AI/인사교_LangChain_20260624


In [7]:
# 관련 라이브러리 설치
!pip install -qU langchain langchain-openai langchain-community langchain-experimental
!pip install -qU langchain-chroma faiss-cpu langchain-teddynote
!pip install -qU langchain-tavily google-search-results pypdf
!pip install -qU mypy

In [4]:
import os
from uuid import uuid4

# 타입 설정
# Sequence : 메시지들의 순서 있는 시퀀스(리스트) 객체 (list 또는 tuple 형태로 메시지를 저장)
# Literal : 값 자체를 타입으로 제한할 수 있도록 해주는 기능
from typing import Annotated, TypedDict, Dict, List, Sequence, Literal
# pydantic : Python 데이터 검증과 설정 관리를 위한 라이브러리
from pydantic import Field, BaseModel, ValidationError

# 상태, 노드
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
# 도구를 노드로 만드는 기능 (자동 맵핑, 상태업데이트, 다중 실행)
from langgraph.prebuilt import ToolNode
# 도구 노드의 결과에 따라 자동으로 라우팅 경로를 설정
from langgraph.prebuilt import tools_condition

# 메모리 저장소
from langgraph.checkpoint.memory import MemorySaver

# 실행 설정
from langchain_core.runnables import RunnableConfig

# 시각화
from IPython.display import Image, display, Markdown
from langchain_teddynote.messages import stream_graph

# 모델 설정
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder
# BaseMessage : BaseMessage를 상속한 다양한 메시지 타입 (HumanMessage, AIMessage, SystemMessage)
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

# 웹 검색 도구
from langchain_community.tools.tavily_search import TavilySearchResults
#from langchain_tavily import TavilySearch

# 도구 정의 데코레이터
from langchain.tools import tool

# RAG 라이브러리
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_1581/3623780955.py:36: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [5]:
# OpenAI key
with open("/content/drive/MyDrive/AI/인사교_LangChain_20260624/key/openai_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['OPENAI_API_KEY'] = api_key

In [6]:
# 웹 검색 도구 Key
with open("/content/drive/MyDrive/AI/인사교_LangChain_20260624/key/tavily_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['TAVILY_API_KEY'] = api_key

- 적응형 RAG 시스템 아키텍처
  - Query 분석과 RAG + self-reflection를 결합
  - Query 분석 : 기존 인덱스(데이터베이스 또는 벡터 저장소)와의 관련성을 기준으로 라우팅
    - [인덱스와 관련 있음]: 쿼리가 인덱스된 콘텐츠와 정렬되면 검색 및 생성을 위해 RAG 모듈로 라우팅
    - [인덱스와 관련 없음]: 쿼리가 인덱스 범위를 벗어나는 경우 웹 검색이나 다른 외부 지식 소스로 라우팅
    - 도메인별 도구나 외부 API와 같이 보다 특수한 시나리오에 대해 라우터 추가

<center>    
<img src="https://arome1004.cafe24.com/images/langchain/langchain_rag06.png" width=70%><br><font size=1>참조 : https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_adaptive_rag/</font>   
</center>  

### 기본 PDF기반 벡터스토어 준비

In [9]:
# 문서로드
loader = PyPDFLoader("/content/drive/MyDrive/AI/인사교_LangChain_20260624/data/SPRi AI Brief_11월호산업동향.pdf")
documents = loader.load()
# 청킹
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap= 50,
)
texts = text_splitter.split_documents(documents)
# 벡터 DB 생성 및 검색기 설정
vec_db = FAISS.from_documents(texts, OpenAIEmbeddings())
retriever = vec_db.as_retriever(search_kwargs={"k": 2})

### Query분석을 위한 Router 만들기

In [14]:
# query와 DB index의 관계를 이진분류하도록 구성
# 관련성이 있으면 vectorstore 없으면 web_search로 나오도록 구성
class RouteQueryResult(BaseModel) :
  ''' 사용자 쿼리를 가장 관련성이 높은 데이터 소스로 라우팅 '''
  datasource : Literal['vectorstore','web_search'] = Field(
      ... , # 필수입력 요소
      description="사용자 질문이 주어지면 웹 검색이나 벡터스토어로 라우팅하도록 선택하세요."
  )


In [15]:
RouteQueryResult(datasource="vectorstore")

RouteQueryResult(datasource='vectorstore')

In [16]:
 # 시스템 메시지와 사용자 질문을 포함한 프롬프트 템플릿 생성
system = """당신은 사용자 질문을 벡터스토어나 웹 검색으로 라우팅하는 전문가입니다.
벡터스토어에는 SPRI AI Bried 11월호 산업동향 관련 문서가 포함되어 있습니다. 주제들은 다음과 같다.
- 미국 민권위원회, 연방정부의 얼굴인식 기술 사용에 따른 민권 영향 분석
- 미국 백악관 예산관리국, 정부의 책임 있는 AI 조달을 위한 지침 발표
- 유로폴, 법 집행에서 AI의 이점과 과제를 다룬 보고서 발간 ·
- OECD, 공공 부문의 AI 도입을 위한 G7 툴킷 발표
- 세계경제포럼, 생성AI 시대의 거버넌스 프레임워크 제시
- CB인사이츠 분석 결과, 2024년 3분기 벤처 투자 31%가 AI 스타트업에 집중
- 메타, 동영상 생성AI 도구 ‘메타 무비 젠’ 공개
- 메타, 이미지와 텍스트 처리하는 첫 멀티모달 AI 모델 ‘라마 3.2’ 공개
- 앨런AI연구소, 벤치마크 평가에서 GPT-4o 능가하는 성능의 오픈소스 LLM ‘몰모’ 공개
- 미스트랄AI, 온디바이스용 AI 모델 ‘레 미니스트로’ 공개
- 카카오, 통합 AI 브랜드 겸 신규 AI 서비스 ‘카나나’ 공개
- 2024년 노벨 물리학상과 화학상, AI 관련 연구자들이 수상
- 미국 국무부, AI 연구에서 국제협력을 위한 ‘글로벌 AI 연구 의제’ 발표
- 일본 AI안전연구소, AI 안전성에 대한 평가 관점 가이드 발간
- 구글 딥마인드, 반도체 칩 레이아웃 설계하는 AI 모델 ‘알파칩’ 발표
- AI21 CEO, AI 에이전트에 트랜스포머 아키텍처의 대안 필요성 강조
- MIT 산업성과센터, 근로자 관점에서 자동화 기술의 영향 조사
- 다이스 조사, AI 전문가의 73%는 2025년 중 이직 고려
- 가트너 예측, AI로 인해 엔지니어링 인력의 80%가 역량 향상 필요
- 인디드 조사 결과, 생성AI가 인간 근로자 대체할 가능성은 희박

이러한 주제에 대한 질문이 있으면 벡터스토어를 사용하세요. 그렇지 않으면 웹 검색을 사용하십시오."""

In [21]:
# 템플릿 생성
route_template = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{query}"),
    ]
)
# 모델 생성
llm = init_chat_model(model="openai:gpt-4o-mini")
# 체인구성하기
question_router = route_template | llm.with_structured_output(RouteQueryResult)


In [22]:
# 테스트
print(question_riuter.invoke("오늘 점심메뉴 추천해줘"))


datasource='web_search'


In [23]:
print(question_riuter.invoke("AI 트렌드 알려줘"))

datasource='vectorstore'
